In [ ]:
import pandas as pd
import numpy as np
import math
import heapq
import distance_metrics

In [ ]:
#Same Iris dataset as the standard KNN implementation. The KD-Tree is built once from
#the training examples and then used to answer nearest-neighbour queries more efficiently
#than brute force, which scans every training example for every query.
df = pd.read_csv('Iris.csv')
x = df.iloc[:, 1:5]
y = df.iloc[:, 5:]

rows, cols = x.shape

shuffle_idx = np.random.permutation(rows)
train_size = int(0.8 * rows)

train_idx = shuffle_idx[:train_size]
test_idx = shuffle_idx[train_size:]

x_train = x.iloc[train_idx].values
x_test = x.iloc[test_idx].values
y_train = y.iloc[train_idx, 0].values
y_test = y.iloc[test_idx, 0].values

print(f"Training samples: {len(x_train)}, Test samples: {len(x_test)}")

In [ ]:
#Node class for the KDTree
class Node:
    def __init__(self, median, coordinate, index):
        #Left and right nodes
        self.left = None
        self.right = None
        #The splitting decision is stored as (dimension_index, median_value), so during
        #search we know which feature to compare and where the split boundary falls.
        self.median = median
        #Coordinate stored by Node
        self.coordinate = coordinate
        self.index = index


root = None

#A KD-Tree recursively partitions the feature space by alternating the splitting dimension
#at each level, cycling through all features. This allows the search to prune entire
#subtrees that cannot contain a closer point than the current best, which is much faster
#than scanning every training example for every query.
def construct_KDTree(training_examples, index, no_features):
    #Base case if no more training examples are there
    if len(training_examples) == 0:
        return None

    #Sorts the current coordinates (essentially parameter vectors) based on an index.
    #Each training example is a tuple where the first element is the coordinate vector
    #and the second is the original index in x_train, so labels can be looked up via y_train.
    sorted_coords = sorted(training_examples, key=lambda x:x[0][index])
    #Obtains the median and where the median index is
    median, median_index = obtain_median(sorted_coords)
    #Constructs node
    node = Node((index, median[0][index]), median[0], median[1])

    #Only creates a recursive call if there is more than 1 coordinate. Otherwise it automatically becomes
    #a child node
    if len(training_examples) > 1:
        node.left = construct_KDTree(sorted_coords[:median_index], (index+1)%no_features, no_features)
        node.right = construct_KDTree(sorted_coords[median_index+1:], (index+1)%no_features, no_features)
    
    
    return node

#Using a tuple model as specified before.
training_examples = [(x_train[i].tolist(), i) for i in range(len(x_train))] 

#Returns a median and the index where the median is.
#Assumption: the sort() method is called first and then this method.
def obtain_median(coords):
    median = int((len(coords) - 1)/2)
    return coords[median], median

#Root of the constructed K-D tree
root = construct_KDTree(training_examples, 0, 4)

In [ ]:
#This traverses the KD-Tree to find the k nearest neighbours of a query point.
#best_dist tracks the worst distance currently in the heap and serves two purposes:
#it is the threshold for admitting new nodes into the heap, and it is the value used
#to decide whether the other subtree is worth exploring at all.
def search(node, query, ind, dimension, best_dist, heap, k):
    #Simply returns the best distance to beat for a node to make it into the heap
    if node is None:
        return best_dist, heap

    #Measuring the current distance using the distance_metrics script
    current_dist = distance_metrics.Euclidean_dist(node.coordinate, query)
    

    #This is to maintain heap of size k at all times which always has the k nearest neighbours
    if len(heap) < k:
        heapq.heappush_max(heap, (current_dist, node.index, node))
        #This is essentially the best_distance to beat for a node
        best_dist = heap[0][0]
    
    #We only want to get rid of the worst element (since this is a max heap) when the 
    #current distance is lesser than the best_distance.
    if len(heap) == k and best_dist > current_dist:
        heapq.heappop_max(heap)
        heapq.heappush_max(heap, (current_dist, node.index, node))
        best_dist = heap[0][0]

    #Deciding which subtree is the primary one (the side the query falls on) and which
    #is the other. We always descend the primary side first since it is more likely to
    #contain the nearest neighbours.
    if query[ind] > node.median[1]:
        primary = node.right
        other = node.left
    else:
        primary = node.left
        other = node.right

    best_dist, heap = search(primary, query, (ind+1)%dimension, dimension, best_dist, heap, k)

    #The perpendicular distance from the query to the splitting hyperplane is
    #abs(query[ind] - node.median[1]). If this is less than best_dist, a closer point
    #could exist on the other side of the split, so we have to explore it too.
    if abs(query[ind] - node.median[1]) < best_dist:
        best_dist, heap = search(other, query, (ind+1)%dimension, dimension, best_dist, heap, k)

    return best_dist, heap


#This predicts one using the kd_tree
def predict_one_kdtree(query, k, dimension):
    heap = []
    heapq.heapify_max(heap)
    best_dist = float('inf')

    best_dist, heap = search(root, query, 0, dimension, best_dist, heap, k)

    #Same technique as a basic KNN: using the largest possible votes
    votes = {}
    for dist, index, node in heap:
        label = y_train[node.index]
        if label not in votes:
            votes[label] = 1
        else:
            votes[label] += 1

    return max(votes, key=lambda x: votes[x])
    
#Creates the prediction on all training examples
def predict_all(testing_examples, testing_labels, k, dimension):
    accuracy = 0
    for i in range(len(testing_examples)):
        if predict_one_kdtree(testing_examples[i], k, dimension) == testing_labels[i]:
            accuracy += 1
    
    return accuracy/len(testing_labels)

print(predict_all(x_test, y_test, 9, 4))